In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV,StratifiedKFold
from mlxtend.evaluate import bootstrap_point632_score
import scipy.stats 

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC 
from mlxtend.data import mnist_data 
from sklearn.metrics import accuracy_score,f1_score 

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score 

In [6]:
X,y = mnist_data()
X = X.astype(np.float32)
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=.2,random_state=1,stratify=y
)

In [7]:
clf1 = LogisticRegression(multi_class='multinomial',solver='newton-cg',random_state=1)
clf2 = KNeighborsClassifier(algorithm='ball_tree',leaf_size=50,n_jobs=-1)
clf3 = DecisionTreeClassifier(random_state=1)
clf4 = SVC(random_state=1)
clf5 = RandomForestClassifier(random_state=1,n_jobs=-1)
pipe1 = Pipeline([
    ('std',StandardScaler()),
    ('clf1',clf1)])
pipe2 = Pipeline([
    ('std',StandardScaler()),
    ('clf2',clf2)])
pipe4 = Pipeline([
    ('std',StandardScaler()),
    ('clf4',clf4)])







In [9]:
param_grid1 = [{
    'clf1__penalty':['l2'],
    'clf1__C':np.power(10.,np.arange(-4,4))
}]
param_grid2 = [{
    'clf2__n_neighbors':np.arange(1,10),
    'clf2__p':[1,2]
}]
param_grid3 = [{
    'max_depth':list(range(1,10))+[None],
    'criterion':['entropy','gini']
}]
param_grid4 = [{
    'clf4__kernel':['rbf'],
    'clf4__C':np.power(10.,np.arange(-4,4))},
    {
    'clf4__kernel':['linear'],
    'clf4__C': np.power(10.,np.arange(-4,4))
    }
]
param_grid5 = [{'n_estimators':[10,100,500,1000,10000]}]


In [ ]:



gridcvs = {}
inner_cv = StratifiedKFold(n_splits=2,shuffle=True,random_state=1)
for pgrid,est,name in zip((param_grid1,param_grid2,param_grid3,param_grid4,param_grid5),(pipe1,pipe2,clf3,pipe4,clf5),('Softmax','KNN','DecisionTree','SVM','RandomForest')):
    gcv = GridSearchCV(
        estimator=est,
        param_grid=pgrid,
        scoring='accuracy',
        cv=inner_cv,
        verbose=0,
        refit=True,
        n_jobs=-1
    )
    gridcvs[name] = gcv
for name,gs_est in sorted(gridcvs.items()):
    print(f'Algorithm : {name}')
    print('inner loop') 
    outer_scores = []
    outer_cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=1)
    for train_idx,valid_idx in outer_cv.split(X_train,y_train):
        gridcvs[name].fit(X_train[train_idx],y_train[train_idx])
        print(f'Best accuracy of inner folds -> {gridcvs[name].best_score_}')
        print(f'best params of inner fold -> {gridcvs[name].best_params_}')
        outer_scores.append(gridcvs[name].best_estimator_.score(X_train[valid_idx],y_train[valid_idx]))
        print(f' Accuracy of outer fold = {outer_scores[-1]*100}')
    print('\n Outer looop')
    print(f'Accuracy = {np.mean(outer_scores)} +- {np.std(outer_scores)}' )    
    


Algorithm : DecisionTree
inner loop
Best accuracy of inner folds -> 0.7259374999999999
best params of inner fold -> {'criterion': 'gini', 'max_depth': None}
 Accuracy of outer fold = 75.5
Best accuracy of inner folds -> 0.7403125
best params of inner fold -> {'criterion': 'entropy', 'max_depth': 7}
 Accuracy of outer fold = 78.25
Best accuracy of inner folds -> 0.73875
best params of inner fold -> {'criterion': 'entropy', 'max_depth': 9}
 Accuracy of outer fold = 77.375
Best accuracy of inner folds -> 0.73375
best params of inner fold -> {'criterion': 'entropy', 'max_depth': 8}
 Accuracy of outer fold = 74.875
Best accuracy of inner folds -> 0.7390625
best params of inner fold -> {'criterion': 'entropy', 'max_depth': 8}
 Accuracy of outer fold = 77.75

 Outer looop
Accuracy = 0.7675000000000001 +- 0.013205112646244242
Algorithm : KNN
inner loop
Best accuracy of inner folds -> 0.88375
best params of inner fold -> {'clf2__n_neighbors': 1, 'clf2__p': 1}
 Accuracy of outer fold = 91.625
Be

d:\ANACONDA\a\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Best accuracy of inner folds -> 0.8890625
best params of inner fold -> {'clf1__C': 0.01, 'clf1__penalty': 'l2'}
 Accuracy of outer fold = 90.0


d:\ANACONDA\a\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Best accuracy of inner folds -> 0.8875
best params of inner fold -> {'clf1__C': 0.01, 'clf1__penalty': 'l2'}
 Accuracy of outer fold = 91.0


d:\ANACONDA\a\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Best accuracy of inner folds -> 0.8928125
best params of inner fold -> {'clf1__C': 0.01, 'clf1__penalty': 'l2'}
 Accuracy of outer fold = 90.0


d:\ANACONDA\a\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Best accuracy of inner folds -> 0.8859375
best params of inner fold -> {'clf1__C': 0.1, 'clf1__penalty': 'l2'}
 Accuracy of outer fold = 89.625


d:\ANACONDA\a\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Best accuracy of inner folds -> 0.8865624999999999
best params of inner fold -> {'clf1__C': 0.01, 'clf1__penalty': 'l2'}
 Accuracy of outer fold = 89.5

 Outer looop
Accuracy = 0.9002500000000001 +- 0.005267826876426378


In [13]:
gcv_model_select = GridSearchCV(
    estimator=clf5,
    param_grid=param_grid5,
    scoring='accuracy',
    n_jobs=-1,
    cv = inner_cv,
    verbose=1,
    refit=True
)
gcv_model_select.fit(X_train,y_train)
print(f'printy accuracy = {gcv_model_select.best_score_}')
print(f'best parameters/model = {gcv_model_select.best_params_}')

Fitting 2 folds for each of 5 candidates, totalling 10 fits
printy accuracy = 0.933
best parameters/model = {'n_estimators': 10000}


In [14]:
train_acc= accuracy_score(y_true=y_train,y_pred = gcv_model_select.predict(X_train))
test_acc = accuracy_score(y_true = y_test,y_pred=gcv_model_select.predict(X_test))


In [15]:
print(f'train acc = {train_acc}')
print(f'test acc = {test_acc}')

train acc = 1.0
test acc = 0.94
